# Homework 4 — Data Preparation

**Dataset:** [BBBC010](https://bbbc.broadinstitute.org/BBBC010), same source as HW 3,
but here we are doing **segmentation**: predict the foreground mask from the brightfield
image. Input = w2 image; target = binary mask.

If `HW_3_sol/data_download/` already exists from HW 3, we symlink the zips to avoid a
second download.

In [ ]:
import os, re, glob, random, urllib.request, zipfile
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
print("Libraries ready.")

## 1. Download

In [ ]:
data_dir = "data_download"
os.makedirs(data_dir, exist_ok=True)

urls = {
    "BBBC010_v2_images.zip":     "https://data.broadinstitute.org/bbbc/BBBC010/BBBC010_v2_images.zip",
    "BBBC010_v1_foreground.zip": "https://data.broadinstitute.org/bbbc/BBBC010/BBBC010_v1_foreground.zip",
}

sibling = "../HW_3_sol/data_download"
for fname, url in urls.items():
    fpath = os.path.join(data_dir, fname)
    if os.path.exists(fpath):
        print(f"  {fname} already present.")
        continue
    sib_path = os.path.join(sibling, fname)
    if os.path.exists(sib_path):
        os.symlink(os.path.abspath(sib_path), fpath)
        print(f"  {fname} linked from HW_3_sol.")
        continue
    print(f"  downloading {fname} ...")
    urllib.request.urlretrieve(url, fpath)
    print("    done.")

## 2. Extract & inspect

In [ ]:
for fname in urls:
    with zipfile.ZipFile(os.path.join(data_dir, fname), "r") as z:
        for m in z.namelist():
            if "__MACOSX" in m or "/.svn" in m or m.endswith(".svn-base"): continue
            z.extract(m, data_dir)

img_files  = sorted(glob.glob(os.path.join(data_dir, "*_w*.tif")))
mask_files = sorted(glob.glob(os.path.join(data_dir, "BBBC010_v1_foreground", "*_binary.png")))
print(f"Images: {len(img_files)}   masks: {len(mask_files)}")

## 3. Pair images (w2) with masks by well code

In [ ]:
WELL_RE = re.compile(r"_([A-P]\d{2})_w(\d)_")

well_to_w2 = {}
for f in img_files:
    m = WELL_RE.search(os.path.basename(f))
    # TODO (Ex 1): keep only channel-2 (brightfield) images
    if m and m.group(2) == ___:
        well_to_w2[m.group(1)] = f

well_to_mask = {os.path.basename(f).replace("_binary.png", ""): f for f in mask_files}
common = sorted(set(well_to_w2) & set(well_to_mask))
print(f"Paired wells: {len(common)}")

## 4. Process & save

* **Image** (input): 16-bit → 8-bit RGB, 512×512 (bilinear)
* **Mask** (target): binary `L` (0/255), 512×512 (nearest)
* 80/20 split (`random.seed(42)`)

```
datasets/bbbc010_seg/
├── train/
│   ├── images/   # 80 brightfield (RGB, 512x512)
│   └── masks/    # 80 binary masks (L, 512x512)
└── test/
    ├── images/
    └── masks/
```

In [ ]:
out_root = "datasets/bbbc010_seg"
for split in ["train", "test"]:
    for sub in ["images", "masks"]:
        os.makedirs(os.path.join(out_root, split, sub), exist_ok=True)

# TODO (Ex 1): shuffle with seed 42, then take the first 80 wells for train
random.seed(___); random.shuffle(common)
train = common[:___]; test = common[___:]
print(f"train={len(train)}  test={len(test)}")

# TODO (Ex 1): target size for the U-Net
TARGET = (___, ___)

def to_uint8(arr):
    arr = arr.astype(np.float32)
    arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)
    return (arr * 255).astype(np.uint8)

def process(well, split):
    img = np.array(Image.open(well_to_w2[well]))
    Image.fromarray(to_uint8(img)).convert("RGB").resize(TARGET, Image.BILINEAR)\
         .save(os.path.join(out_root, split, "images", well + ".png"))
    mask = np.array(Image.open(well_to_mask[well]).convert("L"))
    # TODO (Ex 1): binarize — any pixel >0 → 255, else 0
    binary = ((mask > ___).astype(np.uint8) * ___)
    Image.fromarray(binary, mode="L").resize(TARGET, Image.NEAREST)\
         .save(os.path.join(out_root, split, "masks", well + ".png"))

for w in train: process(w, "train")
for w in test:  process(w, "test")
print("Saved.")

## 5. Verify output

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(6, 8))
train_imgs = sorted(os.listdir(os.path.join(out_root, "train", "images")))
for row, f in enumerate(train_imgs[:3]):
    img = Image.open(os.path.join(out_root, "train", "images", f))
    mask = Image.open(os.path.join(out_root, "train", "masks", f))
    axes[row, 0].imshow(img, cmap="gray");  axes[row, 0].set_title(f"Image (input) — {f[:3]}")
    axes[row, 1].imshow(mask, cmap="gray"); axes[row, 1].set_title("Mask (target)")
    for a in axes[row]: a.axis("off")
plt.suptitle("Train samples — 512×512", fontsize=13)
plt.tight_layout(); plt.show()

for split in ["train", "test"]:
    ni = len(os.listdir(os.path.join(out_root, split, "images")))
    nm = len(os.listdir(os.path.join(out_root, split, "masks")))
    print(f"{split}: images={ni}, masks={nm}")

## Conclusion

Data ready at `datasets/bbbc010_seg/`. Next: `01_TrainModel.ipynb`.